# LFW Grad-CAM — 04. Occlusion faithfulness and report

high-saliency 영역을 가렸을 때 원본 pair cosine이 random 및
low-saliency control보다 더 크게 감소하는지 검사합니다. 이 검증을
통과하지 못하면 Grad-CAM 그림은 논문의 원인 설명 근거로 사용하지
않습니다.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(D:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

MODEL_NAME = "arcface"     # "arcface", "adaface", "magface" 중 이번 실행 모델
MODE = "dev"               # 빠른 검증은 "dev", 전체 논문 실행만 "real"
DATA_FRACTION = 0.10       # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합·tie-break·random control 재현 seed
EXECUTE_STAGE = False      # 입력과 checkpoint를 채운 뒤 실제 계산할 때만 True
WRITE_OUTPUTS = False      # 검증 후 새 artifact를 저장할 때만 True

if MODEL_NAME not in CONFIG["models"]["selected"]:
    raise ValueError(f"지원하지 않는 모델: {MODEL_NAME}")
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")

In [ ]:
import numpy as np
import pandas as pd

from research.embeddings import (
    create_pytorch_adapter_from_spec,
    read_model_spec,
)
from research.explainability.gradcam import (
    occlude_by_saliency,
    occlusion_faithfulness,
)

MODEL_SPEC_PATH = None
CASE_MANIFEST_PATH = None
PAIR_BUNDLE_PATH = None
HEATMAP_INPUT_PATH = None
FAITHFULNESS_OUTPUT_PATH = None
DEVICE = "cpu"
OCCLUSION_FRACTIONS = CONFIG["gradcam"]["faithfulness"]["occlusion_fractions"]
RANDOM_REPEATS = CONFIG["gradcam"]["faithfulness"]["random_repeats"]

In [ ]:
if EXECUTE_STAGE:
    required_paths = [
        MODEL_SPEC_PATH,
        CASE_MANIFEST_PATH,
        PAIR_BUNDLE_PATH,
        HEATMAP_INPUT_PATH,
    ]
    if any(path is None for path in required_paths):
        raise RuntimeError("faithfulness 입력 경로를 모두 지정하세요.")
    spec = read_model_spec(MODEL_SPEC_PATH, verify_checkpoint=True)
    cases = pd.read_parquet(CASE_MANIFEST_PATH)
    pair_bundle = np.load(PAIR_BUNDLE_PATH, allow_pickle=False)
    heatmap_bundle = np.load(HEATMAP_INPUT_PATH, allow_pickle=False)
    expected_ids = cases["case_id"].astype(str).to_numpy()
    if not np.array_equal(pair_bundle["case_id"].astype(str), expected_ids):
        raise ValueError("pair bundle case_id 순서가 다릅니다.")
    if not np.array_equal(heatmap_bundle["case_id"].astype(str), expected_ids):
        raise ValueError("heatmap case_id 순서가 다릅니다.")

    query_images = pair_bundle["query_images"]
    galleries = pair_bundle["gallery_templates"].astype(np.float32)
    gallery_norms = np.linalg.norm(galleries, axis=1, keepdims=True)
    if np.any(gallery_norms <= 0.0):
        raise ValueError("gallery template에 zero norm이 있습니다.")
    galleries = galleries / gallery_norms
    heatmaps = heatmap_bundle["heatmaps"]
    adapter = create_pytorch_adapter_from_spec(spec, device=DEVICE)

    neutral_pixel = np.asarray(
        spec.preprocessing.channel_mean,
        dtype=np.float32,
    )
    if (
        spec.preprocessing.source_color_order
        != spec.preprocessing.model_color_order
    ):
        neutral_pixel = neutral_pixel[::-1]

    def pair_scores(images):
        embeddings = adapter.embed(images).normalized_embedding
        return np.sum(embeddings * galleries, axis=1)

    origin_scores = pair_scores(query_images)
    result_parts = []
    for fraction in OCCLUSION_FRACTIONS:
        high_images = occlude_by_saliency(
            query_images,
            heatmaps,
            fraction=fraction,
            strategy="high_saliency",
            fill_value=neutral_pixel,
            seed=SEED,
        )
        low_images = occlude_by_saliency(
            query_images,
            heatmaps,
            fraction=fraction,
            strategy="low_saliency",
            fill_value=neutral_pixel,
            seed=SEED,
        )
        random_scores = []
        for repeat in range(RANDOM_REPEATS):
            random_images = occlude_by_saliency(
                query_images,
                heatmaps,
                fraction=fraction,
                strategy="random",
                fill_value=neutral_pixel,
                seed=SEED + repeat,
            )
            random_scores.append(pair_scores(random_images))
        random_mean_scores = np.mean(random_scores, axis=0)
        high_scores = pair_scores(high_images)
        low_scores = pair_scores(low_images)
        faith = occlusion_faithfulness(
            origin_scores,
            high_scores,
            control_occluded_scores=random_mean_scores,
        )
        faith.insert(0, "case_id", expected_ids)
        faith["occlusion_fraction"] = float(fraction)
        faith["low_saliency_occluded_score"] = low_scores
        faith["faithfulness_gain_over_low_saliency"] = (
            (origin_scores - high_scores) - (origin_scores - low_scores)
        )
        faith["random_repeats"] = int(RANDOM_REPEATS)
        result_parts.append(faith)
    faithfulness = pd.concat(result_parts, ignore_index=True).merge(
        cases,
        on="case_id",
        how="left",
        validate="many_to_one",
    )
    if WRITE_OUTPUTS:
        if FAITHFULNESS_OUTPUT_PATH is None:
            raise RuntimeError("FAITHFULNESS_OUTPUT_PATH를 지정하세요.")
        destination = Path(FAITHFULNESS_OUTPUT_PATH).resolve()
        if destination.exists():
            raise FileExistsError(f"기존 결과를 덮어쓸 수 없습니다: {destination}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        faithfulness.to_parquet(destination, index=False)
    report = faithfulness.groupby(
        ["occlusion_fraction", "case_group"],
        dropna=False,
    )[
        [
            "saliency_score_drop",
            "faithfulness_gain_over_control",
            "faithfulness_gain_over_low_saliency",
        ]
    ].agg(["count", "mean", "std"])
else:
    report = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
    }
report

보고 시 평균만 제시하지 말고 case 수, 분포, bootstrap 신뢰구간을 함께
계산합니다. 이 노트북의 결과는 정량 압축 성능을 대체하지 않는
후속 분석입니다.